# 06h — Behavioral-structure metrics on the G=28 cohort (local disorganisation / perseveration)

Kickoff **G** of the E→G→F arc (`.claude/plans/coordination-EFG-roadmap.md`); the
recallable runner for **`RESULTS_HANDOFF §16`**. E's §15 order-shuffle-null found **0/15**
order signals — *global* symbol order carries no group signal beyond frequency. G tests the
**orthogonal** hypothesis: *local* execution structure (fragmentation, perseveration,
transition entropy, temporal drift) that the frequency histogram **and** the global-order
tests both miss.

This notebook is a thin driver over the shipped, tested library
(`smartflat.features.symbolic_barycenter.structure_metrics`) — it computes nothing new, it
just re-runs the five public functions on the real cohort and writes the four §16 CSVs. The
metrics are **per-sequence** and vocabulary-agnostic, so the cohort is loaded **ragged**
(true lengths, no resample artifact); length is carried as a covariate because it is itself
group-correlated (RIL administrations are longer).


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe under nbconvert
import numpy as np, pandas as pd
from IPython.display import display
from smartflat.utils.utils_io import get_data_root
from smartflat.features.symbolic_barycenter import vocab
from smartflat.features.symbolic_barycenter.structure_metrics import (
    compute_structure_metrics, structure_group_stats,
    evaluate_incremental_structure, evaluate_structure_length_controlled)
pd.set_option('display.width', 200, 'display.max_columns', 40)

OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'g28', 'structure')
os.makedirs(OUT, exist_ok=True)
RS = 42
print('output dir:', OUT)

In [ ]:
# Canonical G=28 cohort via E's shared loader (06c Cells 2-3). Ragged X: per-sequence
# structure metrics need the true lengths, not an upsampled rectangle.
df, X, labels = vocab.load_g28_cohort()          # X is a ragged list of int sequences
G = int(max(int(np.asarray(s).max()) for s in X)) + 1
print('cohort:', pd.Series(labels).value_counts().to_dict(), '| G =', G,
      '| length range:', min(len(s) for s in X), '-', max(len(s) for s in X))

## (1) Per-sequence metrics table

One row per administration, one float column per metric (transition entropy,
immediate-repeat / switch rate, run-length & dwell stats, fragmentation index, n-gram
coverage, normalised Lempel–Ziv, background fraction, first-vs-second-half drift, length).
NaN-safe: a metric undefined for a sequence yields NaN, never raises.

In [ ]:
metrics_df = compute_structure_metrics(df, seq_col='int_cat_segm_embedding_labels', G=G)
metrics_df.to_csv(os.path.join(OUT, 'structure_metrics_table.csv'), index=False)
print('metrics:', metrics_df.shape)
display(metrics_df.head())

## (2) Descriptive per-metric group stats (§16.2)

Cliff's δ + bootstrap 95% CI, Hedges' g, Mann–Whitney p, and **Benjamini–Hochberg**
`p_value_bh` computed jointly over the full `metric × comparison` family (pre-committed, not
per-comparison). The `spearman_length` column flags metrics whose group effect is really a
length effect — read every metric-level claim against it (see §16.2).

In [ ]:
group_stats = structure_group_stats(metrics_df, labels, random_state=RS)
group_stats.to_csv(os.path.join(OUT, 'structure_group_stats.csv'), index=False)
n_bh = int((group_stats['p_value_bh'] < 0.05).sum())
print(f'BH-significant (metric x comparison): {n_bh} / {len(group_stats)}')
display(group_stats.sort_values('p_value_bh')
        .loc[:, ['metric', 'comparison', 'cliffs_delta', 'cliffs_ci_low', 'cliffs_ci_high',
                 'hedges_g', 'spearman_length', 'p_value_bh']].head(24))

## (3) Incremental held-out AUC over frequency (§16.1)

Leakage-guarded nested CV (shared `baselines._nested_cv_auc`), three feature sets:
`hist` (frequency), `struct` (de-collinearised structure metrics), `both`. The
confirmatory quantity is the paired `both − hist` delta with a bootstrap 95% CI — does
local structure add signal *beyond* frequency?

In [ ]:
inc = evaluate_incremental_structure(X, labels, G, random_state=RS)
inc.to_csv(os.path.join(OUT, 'incremental_structure_summary.csv'), index=False)
display(inc)

## (4) Length-controlled test — the decisive one (§16.3)

Sequence length is group-correlated (RIL longer) and several metrics scale with it, so the
raw §16.1 gain can be a *duration* effect. This reports the incremental AUC of `struct` over
`[hist ⊕ length]` — structure beyond frequency **and** duration. A `delta_struct_given_len`
CI that excludes 0 is the length-robust result (per §16.3: survives on Patient-vs-Control,
does not on HEALTHY-vs-RIL).

In [ ]:
lc = evaluate_structure_length_controlled(X, labels, G, random_state=RS)
lc.to_csv(os.path.join(OUT, 'length_control_summary.csv'), index=False)
display(lc.loc[:, ['comparison', 'classifier', 'auc_hist', 'auc_hist_len',
                   'auc_hist_len_struct', 'delta_struct_given_len',
                   'delta_struct_given_len_ci_low', 'delta_struct_given_len_ci_high']])

## Verdict (reproduces §16.4)

The deliverable is the **library**; the honest finding is a **length-robust positive for
local execution structure on Patient-vs-Control** — patients are more *fragmented* and drift
*less* across task phases — while HEALTHY-vs-RIL reduces to sequence duration and RIL-vs-TBI
is null. Orthogonal to §15 (global order): *frequency dominates global discrimination (E);
local execution structure is a separate, real axis on Patient-vs-Control (G).* Confirm the
four CSVs under `outputs/symbolic_barycenter/g28/structure/` match the §16 tables.